<div style="text-align:center; padding:24px 0 12px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="200"/>
</div>


# GoogleAdsPulse — Paid Media Analytics
## Notebook 2 — SQL Analytics Avancé

### 📝 **VERSION APPRENANT**

---

> **Objectif :** répondre aux 3 questions opérationnelles de Marc-Aurèle avec du
> SQL analytique avancé (DuckDB + JupySQL). Produire les 7 CSV analytiques qui
> alimenteront le dashboard Power BI.

| | |
|---|---|
| **Stack** | DuckDB · JupySQL · pandas · matplotlib · seaborn |
| **Patterns SQL** | `RANK()` · `NTILE()` · `LAG()` · `SUM() OVER ROWS BETWEEN` · `DATE_TRUNC` · `STDDEV()` · self-join |
| **Durée** | 5h à 7h |
| **Sortie** | 7 CSV analytiques |

---

### 📝 Comment utiliser ce notebook

- Chaque étape a son bloc **🎓 MÉTHODE** qui explique le pattern SQL
- Les cellules **📝 TODO** contiennent un squelette SQL à compléter
- Les cellules **🧠 Tes observations** sont pour toi
- ⚠️ Teste chaque étape SQL avant de passer à la suivante


---
## 0. Setup

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine seaborn --quiet


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#F9F9F8',
    'axes.grid':True,'grid.alpha':0.3,'font.size':10,
})

COLORS = {
    'primary':'#534AB7','secondary':'#1D9E75',
    'warning':'#EF9F27','danger':'#E24B4A','neutral':'#888780',
    'blue':'#0EA5E9',
}
CAMPAIGN_COLORS = {
    'Search':'#4285F4','Shopping':'#34A853','Performance Max':'#9333EA',
    'Display':'#FBBC04','Video':'#EA4335',
}
print('✅ Environnement prêt')


### 🎓 MÉTHODE — Pourquoi DuckDB + JupySQL ?

**DuckDB** est une base SQL analytique qui charge directement les CSV — aucun serveur.
Idéale pour les analyses one-shot avec window functions lourdes, 10× plus rapide que pandas.

**JupySQL** permet d'écrire `%%sql df << SELECT ...` dans une cellule Jupyter : le résultat
est directement un DataFrame pandas (via `autopandas = True`), sans passer par
`con.execute(...).df()`.


In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/data/'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/googleadspulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

# Chargement des 5 tables dans DuckDB avec pré-nettoyage minimal
con = duckdb.connect()
con.execute(f"""
    CREATE TABLE accounts   AS SELECT * FROM read_csv_auto('{BASE_URL}accounts.csv');
    CREATE TABLE campaigns  AS SELECT * FROM read_csv_auto('{BASE_URL}campaigns.csv');
    CREATE TABLE ads        AS SELECT * FROM read_csv_auto('{BASE_URL}ads.csv');
    CREATE TABLE keywords   AS SELECT * FROM read_csv_auto('{BASE_URL}keywords.csv');

    -- Performance : nettoyage inline (dédup + filtre anomalies techniques)
    CREATE TABLE perf AS
    SELECT DISTINCT *
    FROM read_csv_auto('{BASE_URL}performance_quotidienne.csv')
    WHERE cost_eur >= 0
      AND clicks <= impressions;
""")


# Volumes
n = con.execute("""
    SELECT 'accounts' AS t, COUNT(*) AS n FROM accounts   UNION ALL
    SELECT 'campaigns',     COUNT(*)      FROM campaigns  UNION ALL
    SELECT 'ads',           COUNT(*)      FROM ads        UNION ALL
    SELECT 'keywords',      COUNT(*)      FROM keywords   UNION ALL
    SELECT 'perf (clean)',  COUNT(*)      FROM perf
""").df()
print(n.to_string(index=False))

# JupySQL
%load_ext sql
%sql con --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback   = False
print('\n%%sql prêt ✅')


---
## Étape 1 — KPIs Overview + variations Period-over-Period (`LAG()`)

### 🎓 MÉTHODE — Calculer les deltas du dashboard 

Les petits `+3.41%` / `-11.42%` à côté des KPIs cards du dashboard sont des **deltas
period-over-period** : comparaison entre la période actuelle et la précédente.

**Pattern SQL :**
```sql
LAG(kpi_value) OVER (ORDER BY periode)  -- valeur période précédente
```

**Ce qu'on livre en Étape 1 :**
1. Les 10 KPIs agrégés **par mois** (24 périodes)
2. Le delta Month-over-Month pour chaque KPI (% de variation)
3. Une vue récapitulative "période actuelle vs précédente" pour alimenter les cards Power BI


In [ ]:
%%sql df_kpi_mensuel <<
-- 📝 TODO — Étape 1 : KPIs mensuels + deltas MoM avec LAG()
--
-- 1. CTE 'mensuel' : agrège les 10 KPIs par mois
--    - strftime(date, '%Y-%m') pour obtenir 'YYYY-MM'
--    - 10 KPIs : impressions, clicks, conversions (SUM)
--    -           ctr, cpc, cpm, cost_per_conv, conv_rate (ratios avec NULLIF)
--    -           spend, conv_value, roas
--
-- 2. Query finale : SELECT * + LAG() pour chaque KPI du mois précédent
--    LAG(kpi) OVER (ORDER BY mois) AS kpi_prev
--
-- 3. Calcule les deltas % pour spend, clicks, conversions, roas :
--    (current - prev) * 100.0 / NULLIF(prev, 0)

-- TODO


In [ ]:
df_kpi_mensuel.tail(6)

In [ ]:
# 📝 TODO — Visualiser l'évolution des 4 KPIs clés avec annotation des deltas
#
# fig, axes = plt.subplots(2, 2, figsize=(15, 9))
# 4 subplots : Spend, Clicks, Conversions, ROAS
#
# Pour chaque KPI :
# 1. Bar chart avec df_kpi_mensuel filtré (dropna sur delta_spend_pct)
# 2. Annoter chaque barre avec son delta % (vert si positif, rouge si négatif)
#
# 💡 ax.annotate(f'{dval:+.0f}%', xy=(i, row[kpi]), xytext=(0, 3), textcoords='offset points')
# 💡 FuncFormatter pour axe Y

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## Étape 2 — Top & Bottom campaigns avec `RANK()` + `NTILE()`

### 🎓 MÉTHODE — Classer les campagnes sur plusieurs axes

Une campagne peut être **#1 en spend** (très visible) mais **dernière en ROAS** (très peu
rentable). Le classement multidimensionnel révèle les arbitrages.

**2 patterns SQL complémentaires :**

```sql
-- RANK : classement ordinal (1, 2, 3, ..., N)
RANK() OVER (ORDER BY roas DESC) AS rang_roas

-- NTILE(4) : segmentation en 4 quartiles de performance
NTILE(4) OVER (ORDER BY roas DESC) AS quartile_roas
-- Quartile 1 = top 25%, Quartile 4 = bottom 25%
```

NTILE est particulièrement utile pour les Power BI dashboards : il permet de colorer
automatiquement les campagnes par quartile (vert/jaune/orange/rouge).


In [ ]:
%%sql df_campaigns_rank <<
-- 📝 TODO — Étape 2 : Top/Bottom campaigns avec RANK() + NTILE(4)
--
-- 1. CTE 'perf_camp' : par campaign_id (JOIN perf + campaigns)
--    - campaign_id, campaign_name, campaign_type, account_id
--    - impressions, clicks, conversions, spend, conv_value (SUM)
--    - ctr, cpc, cost_per_conv, roas (ratios NULLIF)
--
-- 2. Query finale : ajoute 3 RANK et 2 NTILE(4) :
--    RANK() OVER (ORDER BY roas DESC NULLS LAST) AS rang_roas
--    RANK() OVER (ORDER BY spend DESC) AS rang_spend
--    RANK() OVER (ORDER BY cost_per_conv ASC NULLS LAST) AS rang_cpa
--    NTILE(4) OVER (ORDER BY roas DESC NULLS LAST) AS quartile_roas
--    NTILE(4) OVER (ORDER BY spend DESC) AS quartile_spend
--
-- 3. Flag performance via CASE :
--    quartile_roas = 1 → 'Top 25% ROAS'
--    quartile_roas = 4 → 'Bottom 25% ROAS'
--    sinon             → 'Moyen'

-- TODO


In [ ]:
df_campaigns_rank[['campaign_name','campaign_type','spend','conversions','roas','rang_roas','quartile_roas','flag_performance']].head(10)

In [ ]:
# 📝 TODO — 2 panels : Top 10 par ROAS + répartition quartiles par campaign_type
#
# Panel 1 — Barh Top 10 :
#   top10 = df_campaigns_rank.nlargest(10, 'roas').sort_values('roas')
#   Couleurs par campaign_type (CAMPAIGN_COLORS)
#   Ligne verticale rouge en x=3 (seuil rentabilité)
#
# Panel 2 — Stacked bar chart :
#   Groupby campaign_type + quartile_roas puis .size().unstack()
#   .plot(kind='bar', stacked=True)

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## Étape 3 — Rolling 7-day average (`SUM() OVER ROWS BETWEEN`)

### 🎓 MÉTHODE — Lisser les variations quotidiennes

Les données quotidiennes oscillent fortement (weekend vs semaine, jours fériés, etc.).
Le **rolling 7-day average** lisse ces oscillations pour révéler la vraie tendance.

**Pattern SQL puissant :**
```sql
SUM(cost_eur) OVER (
    ORDER BY date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
) / 7.0 AS cost_rolling_7d
```

La clause `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` définit une **fenêtre glissante**
de 7 lignes (jour courant + 6 précédents). C'est exactement le type de calcul que
Power BI fait pour ses graphiques lissés — mais en SQL, c'est gratuit et performant.


In [ ]:
%%sql df_rolling <<
-- 📝 TODO — Étape 3 : Rolling 7-day average avec SUM() OVER ROWS BETWEEN
--
-- 1. CTE 'daily' : agrège par date (SUM impressions, clicks, conversions, cost, conv_value)
--
-- 2. Query finale, ajouter 3 calculs :
--
--    Rolling 7 jours :
--      AVG(spend) OVER (
--        ORDER BY date
--        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
--      ) AS spend_ma7
--      (idem pour clicks_ma7)
--
--    Rolling 30 jours :
--      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW → spend_ma30
--
--    Cumul depuis le début :
--      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW → spend_cumul

-- TODO


In [ ]:
df_rolling.tail(5)

In [ ]:
# 📝 TODO — Visualiser spend quotidien + MA7 + MA30
#
# fig, ax = plt.subplots(figsize=(15, 6))
# 1. ax.bar() : spend brut en barres grises alpha 0.3
# 2. ax.plot() : spend_ma7 en ligne bleue épaisse
# 3. ax.plot() : spend_ma30 en ligne rouge plus épaisse
# Légende claire, titre, FuncFormatter

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## Étape 4 — Cohort Analysis des keywords (`DATE_TRUNC` + `DATEDIFF` + pivot)

### 🎓 MÉTHODE — Est-ce que mes keywords tiennent dans le temps ?

Un mot-clé peut performer brillamment pendant 2 mois puis s'essouffler (saturation du marché,
nouveau concurrent, évolution de l'intention). La cohort analysis keywords permet de voir
**combien de keywords d'une cohorte continuent à performer dans les mois suivants**.

**Principe :**
- Une **cohorte = tous les keywords qui ont reçu leur premier clic le même mois**
- Pour chaque cohorte, on mesure la **rétention de performance** (keywords avec clicks > 0)
  dans les mois suivants

**Pattern SQL :**
```sql
DATE_TRUNC('month', MIN(date))  -- mois du 1er clic = cohort
DATEDIFF('month', cohort_mois, snapshot_mois) AS mois_relatif
```


In [ ]:
%%sql df_cohort_raw <<
-- 📝 TODO — Étape 4 : Cohort analysis keywords (DATE_TRUNC + DATEDIFF)
--
-- 1. CTE 'first_click' : pour chaque keyword, trouve le mois de son 1er clic
--    SELECT keyword_id, DATE_TRUNC('month', MIN(date)) AS cohort_mois
--    FROM perf WHERE keyword_id IS NOT NULL AND clicks > 0
--    GROUP BY keyword_id
--
-- 2. CTE 'kw_monthly' : pour chaque keyword × mois, vérifie s'il est actif (clicks > 0)
--    GROUP BY keyword_id, DATE_TRUNC('month', date)
--    HAVING SUM(clicks) > 0
--
-- 3. Query finale : join + calcul mois relatif
--    DATEDIFF('month', fc.cohort_mois, km.snapshot_mois) AS mois_relatif
--    WHERE mois_relatif BETWEEN 0 AND 12
--    GROUP BY cohort_mois, mois_relatif
--    COUNT(DISTINCT keyword_id) → nb_keywords_actifs

-- TODO


In [ ]:
# 📝 TODO — Pivoter le résultat cohort en table triangulaire
#
# 1. Convertis cohort_mois en string 'YYYY-MM' (→ cohort_mois_str)
# 2. Taille initiale de chaque cohorte = df où mois_relatif==0, indexé par cohort_mois_str
# 3. Pivot :
#    cohort_pivot = df.pivot_table(index='cohort_mois_str',
#                                   columns='mois_relatif',
#                                   values='nb_keywords_actifs')
# 4. Conversion en % : cohort_ret = cohort_pivot.divide(cohort_size, axis=0) * 100

# TODO
cohort_ret = None

if cohort_ret is not None:
    print('Table cohort — rétention keywords (%) :')
    print(cohort_ret.fillna('').to_string())


In [ ]:
# 📝 TODO — Heatmap triangulaire des cohortes
#
# sns.heatmap(cohort_ret,
#             annot=True, fmt='.0f',
#             cmap='RdYlGn', vmin=0, vmax=100,
#             linewidths=0.3, linecolor='white')
# Figsize (13, 8), titre, labels axes

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## Étape 5 — Détection d'anomalies par Z-score (`AVG OVER` + `STDDEV OVER`)

### 🎓 MÉTHODE — Automatiser la détection des dérapages

Marc-Aurèle ne peut pas surveiller manuellement 25 campagnes chaque jour. On automatise
avec un **z-score** : combien d'écart-types le spend du jour s'écarte-t-il de la moyenne
historique de la campagne ?

**Formule :**
```
z = (spend_jour - moyenne_30j) / ecart_type_30j
```

**Interprétation :**
- `|z| < 1` → normal
- `|z| entre 1 et 2` → à surveiller
- `|z| > 2` → **anomalie** (5% de probabilité en loi normale)
- `|z| > 3` → anomalie sévère (0.3% de probabilité)

**Pattern SQL entièrement dans une window function :**
```sql
(spend - AVG(spend) OVER (PARTITION BY campaign_id ORDER BY date
                          ROWS BETWEEN 29 PRECEDING AND 1 PRECEDING))
/ NULLIF(STDDEV(spend) OVER (...), 0)
```

Notez le `BETWEEN 29 PRECEDING AND 1 PRECEDING` — on **exclut le jour courant** du calcul
de la moyenne (sinon il "vote" pour lui-même).


In [ ]:
%%sql df_anomalies <<
-- 📝 TODO — Étape 5 : Détection anomalies par z-score
--
-- 1. CTE 'daily_camp' : agrège par (date, campaign_id) : spend_jour, clicks_jour, conv_jour
--
-- 2. CTE 'with_zscore' : calcule moyenne et écart-type des 30 jours PRÉCÉDENTS
--    ⚠️ IMPORTANT : ROWS BETWEEN 29 PRECEDING AND 1 PRECEDING (exclut le jour courant)
--    AVG(spend_jour) OVER (PARTITION BY campaign_id ORDER BY date ROWS BETWEEN 29 PRECEDING AND 1 PRECEDING) AS moyenne_30j
--    STDDEV(spend_jour) OVER (... mêmes clauses) AS ecart_type_30j
--
-- 3. Query finale : z_score + niveau_alerte (CASE WHEN)
--    z_score = (spend_jour - moyenne_30j) / NULLIF(ecart_type_30j, 0)
--    Niveaux :
--      |z| > 3 → '🔴 Anomalie sévère'
--      |z| > 2 → '🟠 Anomalie'
--      |z| > 1 → '🟡 À surveiller'
--      else   → '✅ Normal'
--    WHERE moyenne_30j IS NOT NULL AND ecart_type_30j IS NOT NULL
--    ORDER BY ABS(z_score) DESC NULLS LAST LIMIT 20

-- TODO


In [ ]:
df_anomalies

In [ ]:
# 📝 TODO — Stats résumées : combien d'observations par niveau ?
#
# Utilise pd.read_sql(""" ... """, con) pour une nouvelle requête qui :
# - Recalcule le z-score (même logique)
# - GROUP BY niveau (CASE WHEN ABS(z) > 3 THEN ... etc.)
# - COUNT(*) AS nb_observations

# TODO
summary = None

if summary is not None:
    print('Répartition des observations par niveau :')
    print(summary.to_string(index=False))


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## Étape 6 — Breakdown Device × Jour × Heure (pivots SQL)

### 🎓 MÉTHODE — Pivot avec `CASE WHEN` agrégés

DuckDB supporte le mot-clé `PIVOT`, mais le pattern **SQL portable** (qui marche partout)
utilise des `SUM(CASE WHEN ... THEN ... END)` :

```sql
SELECT
    dimension_y,
    SUM(CASE WHEN dim_x = 'Desktop' THEN valeur ELSE 0 END) AS desktop,
    SUM(CASE WHEN dim_x = 'Mobile'  THEN valeur ELSE 0 END) AS mobile,
    SUM(CASE WHEN dim_x = 'Tablet'  THEN valeur ELSE 0 END) AS tablet
FROM ...
GROUP BY dimension_y
```

On livre deux pivots en Étape 6 :
1. **Device × Campaign Type** : où convertit-on le mieux ?
2. **Heure × Jour de semaine** : quand concentrer le budget ?


In [ ]:
%%sql df_device_type <<
-- 📝 TODO — Étape 6a : Pivot Device × Campaign Type (CASE WHEN agrégés)
--
-- Structure :
-- SELECT
--   c.campaign_type,
--   SUM(p.clicks) AS clicks_total,
--
--   -- Pivot clicks par device
--   SUM(CASE WHEN p.device = 'Desktop' THEN p.clicks ELSE 0 END) AS clicks_desktop,
--   SUM(CASE WHEN p.device = 'Mobile'  THEN p.clicks ELSE 0 END) AS clicks_mobile,
--   SUM(CASE WHEN p.device = 'Tablet'  THEN p.clicks ELSE 0 END) AS clicks_tablet,
--
--   -- CVR par device (ratio conversions/clicks par device)
--   ROUND(SUM(CASE WHEN device='Desktop' THEN conversions ELSE 0 END) * 100.0
--         / NULLIF(SUM(CASE WHEN device='Desktop' THEN clicks ELSE 0 END), 0), 2) AS cvr_desktop_pct,
--   -- idem pour Mobile et Tablet
--
-- FROM perf p JOIN campaigns c USING(campaign_id)
-- GROUP BY c.campaign_type
-- ORDER BY clicks_total DESC

-- TODO


In [ ]:
df_device_type

In [ ]:
%%sql df_jour_heure <<
-- 📝 TODO — Étape 6b : Heatmap Jour × Heure
--
-- Agréger par (day_of_week, hour) :
--   SUM(cost_eur), SUM(clicks), SUM(conversions)
--   cvr_pct = SUM(conversions) * 100.0 / NULLIF(SUM(clicks), 0)
--
-- Trier pour respecter l'ordre Lundi → Dimanche :
--   ORDER BY CASE day_of_week WHEN 'Monday' THEN 1 WHEN 'Tuesday' THEN 2 ... END, hour

-- TODO


In [ ]:
# 📝 TODO — Visualisations du breakdown
#
# fig, axes = plt.subplots(1, 2, figsize=(17, 6))
#
# Panel 1 — Heatmap jour × heure (spend)
#   Pivot df_jour_heure : index='day_of_week', columns='hour', values='spend'
#   Réindexer selon l'ordre ['Monday','Tuesday',...,'Sunday']
#   sns.heatmap(pivot, cmap='YlOrRd')
#
# Panel 2 — CVR par device × campaign_type
#   Barplot groupé depuis df_device_type
#   Colonnes : cvr_desktop_pct, cvr_mobile_pct, cvr_tablet_pct

# TODO


### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## Étape 7 — Self-join : comparer chaque campagne à son benchmark account

### 🎓 MÉTHODE — Self-join pour comparaison à une moyenne

Un ROAS de 4× est-il bon ou mauvais ? **Ça dépend du client.** Pour TechShop CI
(e-commerce à marge faible), 4× est correct. Pour BankAfrica SaaS (B2B à forte marge),
c'est médiocre. Le vrai benchmark est **la moyenne de l'account lui-même**.

**Pattern : self-join entre une vue agrégée par campagne et une vue agrégée par account :**
```sql
WITH perf_camp  AS (SELECT account_id, campaign_id, AVG(roas) ...),
     perf_acc   AS (SELECT account_id,              AVG(roas) ...)
SELECT c.*, (c.roas / a.roas_account - 1) * 100 AS ecart_vs_benchmark_pct
FROM perf_camp c
JOIN perf_acc  a USING(account_id)
```
Chaque campagne reçoit son **écart en % par rapport à la moyenne de son account**.


In [ ]:
%%sql df_benchmark <<
-- 📝 TODO — Étape 7 : Self-join campagne vs benchmark account
--
-- 1. CTE 'perf_camp' : par campaign_id × account_id
--    spend, conv_value, roas_camp, cpa_camp
--
-- 2. CTE 'perf_acc' : par account_id (benchmark)
--    roas_account, cpa_account
--
-- 3. Query finale : JOIN USING(account_id)
--    ecart_roas_pct = (roas_camp - roas_account) * 100.0 / NULLIF(roas_account, 0)
--    Verdict (CASE WHEN) :
--      roas_camp > roas_account * 1.2 → '🟢 Surperforme'
--      roas_camp < roas_account * 0.8 → '🔴 Sous-performe'
--      sinon                          → '🟡 Dans la norme'
--    JOIN aussi avec accounts pour avoir account_name
--    ORDER BY ecart_roas_pct DESC NULLS LAST

-- TODO


In [ ]:
df_benchmark[['account_name','campaign_name','campaign_type','roas_camp','roas_account','ecart_roas_pct','verdict']].head(15)

### 🧠 Tes observations

*Écris ici en 3-5 lignes ce que tu observes dans les résultats ci-dessus.*

- 
- 
- 


---
## 8. Export des 7 CSV analytiques pour Power BI

In [ ]:
# 📝 TODO — Export des 7 CSV analytiques pour Power BI (NB3)
#
# Sauvegarde les 7 DataFrames dans SAVE_PATH :
#   df_kpi_mensuel        → gads_kpi_mensuel.csv
#   df_campaigns_rank     → gads_campaigns_rank.csv
#   df_rolling            → gads_rolling.csv
#   cohort_ret            → gads_cohort_keywords.csv
#   df_anomalies          → gads_anomalies.csv
#   df_jour_heure         → gads_jour_heure.csv
#   df_benchmark          → gads_benchmark.csv
#
# 💡 .to_csv(f'{SAVE_PATH}nom.csv', index=False)

# TODO

print('✅ 7 CSV exportés dans', SAVE_PATH)


---
## Bilan du Notebook 2

### Réponses aux 3 questions de Marc-Aurèle

| Question | Réponse | Étape |
|---|---|---|
| Quelles campagnes sont vraiment rentables (pas juste en CPC) ? | RANK + NTILE par ROAS + verdict vs benchmark | **2 + 7** |
| Comment détecter les dérapages automatiquement ? | Z-score sur 30j glissants — alerte dès `|z| > 2` | **5** |
| Mes keywords tiennent-ils dans le temps ? | Cohort analysis de rétention | **4** |

### Patterns SQL maîtrisés (13 patterns distincts)

| # | Pattern | Étape |
|---|---|---|
| 1 | `LAG()` pour deltas period-over-period | 1 |
| 2 | `RANK()` / `DENSE_RANK()` ordinal | 2 |
| 3 | `NTILE(4)` quartiles | 2 |
| 4 | `SUM() OVER (ROWS BETWEEN N PRECEDING AND CURRENT ROW)` | 3 |
| 5 | `AVG() OVER (ROWS BETWEEN ... )` rolling | 3, 5 |
| 6 | `SUM() OVER (ROWS BETWEEN UNBOUNDED PRECEDING ...)` cumul | 3 |
| 7 | `DATE_TRUNC('month', ...)` | 4, 1 |
| 8 | `DATEDIFF('month', ...)` | 4 |
| 9 | `STDDEV() OVER` pour z-score | 5 |
| 10 | `CASE WHEN ... THEN ... END` agrégé pour pivots | 6 |
| 11 | CTEs multi-niveaux (`WITH a AS ..., b AS ...`) | 1, 2, 5, 7 |
| 12 | Self-join via CTEs séparées + JOIN USING | 7 |
| 13 | `NULLIF(x, 0)` protection division zéro | partout |

### 7 CSV exportés pour le NB3 Power BI

| Fichier | Usage Power BI |
|---|---|
| `gads_kpi_mensuel.csv` | KPI cards avec deltas — page Overview |
| `gads_campaigns_rank.csv` | Tableau Top/Bottom — page Campaigns |
| `gads_rolling.csv` | Timeline lissée — page Overview |
| `gads_cohort_keywords.csv` | Heatmap triangulaire — page Keywords |
| `gads_anomalies.csv` | Alertes automatiques — page Overview ou sidebar |
| `gads_jour_heure.csv` | Heatmap heure × jour — page Breakdown |
| `gads_benchmark.csv` | Verdict par campagne — page Campaigns |

### Pour le NB3

- Ingérer les 7 CSV dans Power BI
- Construire l'architecture en étoile
- Écrire 30+ mesures DAX avec deltas period-over-period
- Maquetter les 5 pages (Overview, Campaigns, Keywords, Conversions, Breakdown) style 


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.
